In [ ]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_neo4j import Neo4jGraph, Neo4jVector
from langchain_ollama import OllamaEmbeddings
from langchain_ollama import ChatOllama

embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)


llm = ChatOllama(
    model="qwen3:0.6b",
    temperature=0
)
load_dotenv()

/tmp/ipykernel_11847/201939007.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/tmp/ipykernel_11847/201939007.py:5: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.graph_transformers import LLMGraphTransformer


True

In [2]:
# PyPDFLoader yields one Document per page
loader = PyPDFLoader("./elon_musk.pdf")
pages = loader.load()

for i, p in enumerate(pages):
    print(f"Page {i + 1}: {len(p.page_content)} chars")

Page 1: 2343 chars
Page 2: 1192 chars


In [3]:
# smaller chunks give the LLM tighter context for entity extraction
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(pages)

print(f"{len(chunks)} chunks created")

14 chunks created


In [4]:
graph = Neo4jGraph(
    url=os.environ["NEO4J_URI"],
    username=os.environ["NEO4J_USERNAME"],
    password=os.environ["NEO4J_PASSWORD"],
    database="5a5aa232",
)

In [5]:
graph_transformer = LLMGraphTransformer(llm=llm)

In [6]:
graph_docs = graph_transformer.convert_to_graph_documents(chunks)

print(f"{len(graph_docs)} graph documents extracted")

# spot-check the first extraction
print("Nodes:", [n.id for n in graph_docs[0].nodes])
print("Rels: ", [(r.source.id, r.type, r.target.id) for r in graph_docs[0].relationships])

14 graph documents extracted
Nodes: ['Overview', 'Family_Background']
Rels:  [('Overview', 'OVERVIEW', 'Overview')]


In [7]:
# include_source=True links each entity node back to its source Document node,
# which is required for Neo4jVector.from_existing_graph in the next cell
graph.add_graph_documents(
    graph_docs,
    include_source=True,
    baseEntityLabel=True
)
print("Graph stored in Neo4J")

Graph stored in Neo4J


In [8]:
# create a vector index over the Document nodes stored above
vector_index = Neo4jVector.from_existing_graph(
    embedding=embeddings,
    url=os.environ["NEO4J_URI"],
    username=os.environ["NEO4J_USERNAME"],
    password=os.environ["NEO4J_PASSWORD"],
    database="5a5aa232",  # <-- ye missing tha
    index_name="elon_musk_chunks",
    node_label="Document",
    text_node_properties=["text"],
    embedding_node_property="embedding",
)

print("Vector index created")

Vector index created


In [9]:
# verify what landed in Neo4J
node_counts = graph.query(
    "MATCH (n) RETURN labels(n) AS label, count(n) AS count ORDER BY count DESC"
)
rel_counts = graph.query(
    "MATCH ()-[r]->() RETURN type(r) AS type, count(r) AS count ORDER BY count DESC"
)
print("Nodes:")
for r in node_counts:
    print(" ", r)
print("Relationships:")
for r in rel_counts:
    print(" ", r)

Nodes:
  {'label': ['Document'], 'count': 14}
  {'label': ['__Entity__', 'Person'], 'count': 13}
  {'label': ['__Entity__', 'Entity'], 'count': 9}
  {'label': ['__Entity__', 'Location'], 'count': 5}
  {'label': ['__Entity__', 'Year'], 'count': 4}
  {'label': ['__Entity__', 'Child'], 'count': 3}
  {'label': ['__Entity__', 'Company'], 'count': 2}
  {'label': ['__Entity__', 'Product'], 'count': 2}
  {'label': ['__Entity__', 'Overview'], 'count': 1}
  {'label': ['__Entity__', 'Family background /think'], 'count': 1}
  {'label': ['__Entity__', 'Entity', 'Company'], 'count': 1}
  {'label': ['__Entity__', 'Person', 'Entity'], 'count': 1}
  {'label': ['__Entity__', 'City'], 'count': 1}
  {'label': ['__Entity__', 'State'], 'count': 1}
  {'label': ['__Entity__', 'Entity', 'Country'], 'count': 1}
  {'label': ['__Entity__', 'Role'], 'count': 1}
  {'label': ['__Entity__', 'Entity', 'Company', 'Services'], 'count': 1}
  {'label': ['__Entity__', 'Service'], 'count': 1}
  {'label': ['__Entity__', 'Org